# In-Silico Control: Steering Between Target FC Regimes

This notebook performs a substantive *in-silico control* experiment on the LSD pharmacological dataset. For every trained architecture we:

1. **Steer between two target FC regimes** — Placebo and LSD — by optimising the constant-in-time control input $u$ so that the simulated FC matches each condition.
2. **Compare the optimised control inputs $u^\star$** across architectures, both numerically and visually.
3. **Probe robustness** by transferring each model's $u^\star$ to the other architectures; we focus on pairs that are statistically indistinguishable on FC and FCD (Kolmogorov–Smirnov two-sample test, Holm–Bonferroni corrected).

All models are loaded from the checkpoints in `checkpoints/` (learned on the same data), so their *structure* is fixed; we only optimise the external control $u \in \mathbb{R}^{m}$.

In [1]:
import os
import sys
import json
import time
from pathlib import Path

notebook_dir = Path().absolute()
if notebook_dir.name == "examples":
    os.chdir(notebook_dir.parent)

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401
from scipy.stats import ks_2samp

plt.style.use(["science", "no-latex"])
%matplotlib inline

## 1. Configuration

The dataset is small (n_rois = 4, TR = 2 s) which keeps the control optimisation affordable on CPU. Adjust `N_OPT_STEPS`, `N_SIM_STEPS` and `BATCH_SIZE` to trade speed for variance.

In [2]:
DATASET_TYPE = "lsd"
LSD_DATA_DIR = "data/lsd"
CHECKPOINT_DIR = "checkpoints"
OUT_DIR_PDF = Path("paper/images/lsd")
OUT_DIR_PNG = Path("paper/images_png/lsd")
OUT_DIR_SVG = Path("paper/images_svg/lsd")
RESULTS_PATH = Path("results/lsd_in_silico_control.json")
for d in (OUT_DIR_PDF, OUT_DIR_PNG, OUT_DIR_SVG):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Simulation settings
DT = 0.72
DT_MIN = 0.05
N_SIM_STEPS = 120       # short rollout to keep optimisation fast
BATCH_SIZE = 16         # independent noise realisations

# Optimisation settings
N_OPT_STEPS = 60
LR = 5e-2
REG_L2 = 1e-3
FC_LOSS_WEIGHT = 1.0

# Evaluation (post-optimisation)
N_EVAL_BATCHES = 8       # total evaluation trajectories = BATCH_SIZE * N_EVAL_BATCHES
N_EVAL_STEPS = 240       # full LSD time length

RANDOM_SEED = 0

MODEL_CHECKPOINTS = {
    "Hopf": "lsd_hopf.pt",
    "GNN Hopf": "lsd_gnn_hopf.pt",
    "Hybrid Hopf": "lsd_hybrid_hopf.pt",
    "Hybrid+Neural": "lsd_hybrid_neural.pt",
    "Neural SDE": "lsd_nsde.pt",
}

TARGET_CONDITIONS = ["Placebo", "LSD"]

print(f"Device: {DEVICE}")
print(f"Simulation: dt={DT}, n_steps={N_SIM_STEPS}, batch={BATCH_SIZE}")
print(f"Optimisation: n_steps={N_OPT_STEPS}, lr={LR}, reg_l2={REG_L2}")

Device: cuda
Simulation: dt=0.72, n_steps=120, batch=16
Optimisation: n_steps=60, lr=0.05, reg_l2=0.001


## 2. Load Data and Define Target FC Regimes

We treat the empirical **Placebo** mean FC and the empirical **LSD** mean FC as the two regimes we wish to steer the models between. We also store the raw per-subject FC matrices so that statistical tests downstream compare distributions rather than point estimates.

In [3]:
from src.dataset import load_lsd_data
from src.dataset.data_loader import compute_fc_from_timeseries

ts_np, ctrl_np, patient_ids, cond_names = load_lsd_data(LSD_DATA_DIR)
ts_t = torch.from_numpy(ts_np).float()
n_subjects, n_rois, n_timepoints = ts_t.shape

# Robust condition labeling from the (LSD, Ket) indicator vectors
condition_of_subject = []
for row in ctrl_np:
    if tuple(row) == (0.0, 0.0):
        condition_of_subject.append("Placebo")
    elif tuple(row) == (1.0, 0.0):
        condition_of_subject.append("LSD")
    else:
        condition_of_subject.append("LSD+Ket")
condition_of_subject = np.array(condition_of_subject)

empirical_fc_all = compute_fc_from_timeseries(ts_t)  # (n_subjects, n_rois, n_rois)

empirical_fc_by_cond = {}
empirical_fc_mean = {}
for cond in TARGET_CONDITIONS:
    mask = condition_of_subject == cond
    fcs = empirical_fc_all[mask]
    empirical_fc_by_cond[cond] = fcs
    empirical_fc_mean[cond] = fcs.mean(dim=0)
    print(f"{cond:8s}: {mask.sum():3d} subjects, FC mean |off-diag| = {(empirical_fc_mean[cond] - torch.eye(n_rois)).abs().mean().item():.3f}")

# Static pooled empirical FCD for later statistical comparisons
def fcd_windowed(ts: torch.Tensor, window: int = 30, step: int = 3) -> torch.Tensor:
    """Compute sliding-window FCD upper-triangle entries for a batch (B, n_rois, T)."""
    ts_real = ts.real if torch.is_complex(ts) else ts
    B, R, T = ts_real.shape
    idx = torch.triu_indices(R, R, offset=1)
    windows = []
    for s in range(0, T - window + 1, step):
        w = ts_real[:, :, s:s + window]
        w = w - w.mean(dim=2, keepdim=True)
        w = w / (w.std(dim=2, keepdim=True) + 1e-8)
        fc_w = torch.bmm(w, w.transpose(1, 2)) / (window - 1)
        windows.append(fc_w[:, idx[0], idx[1]])
    # windows: (W, B, n_edges) -> (B, W, n_edges)
    V = torch.stack(windows, dim=0).permute(1, 0, 2)
    # Dynamic FC: window-vs-window correlation, flatten upper triangle
    V = V - V.mean(dim=2, keepdim=True)
    V = V / (V.norm(dim=2, keepdim=True) + 1e-8)
    fcd = torch.bmm(V, V.transpose(1, 2))  # (B, W, W)
    Wn = fcd.shape[1]
    tri = torch.triu_indices(Wn, Wn, offset=1)
    return fcd[:, tri[0], tri[1]]  # (B, n_pairs)

empirical_fcd_by_cond = {
    cond: fcd_windowed(ts_t[condition_of_subject == cond]) for cond in TARGET_CONDITIONS
}
for cond in TARGET_CONDITIONS:
    print(f"{cond:8s}: empirical FCD samples = {empirical_fcd_by_cond[cond].numel()}")

Placebo :  25 subjects, FC mean |off-diag| = 0.184
LSD     :  25 subjects, FC mean |off-diag| = 0.100
Placebo : empirical FCD samples = 62125
LSD     : empirical FCD samples = 62125


## 3. Load Trained Models

Each model was trained with one or two auxiliary control nodes (see `n_control_dims` in the checkpoint config). We freeze all model parameters: the only trainable degrees of freedom in what follows are the $u$ vectors.

In [13]:
%%bash
find /usr/share /usr/local /etc /opt -name "modules.sh" -o -name "bash" -path "*/init/bash" 2>/dev/null | grep "init/bash\|modules.sh"

CalledProcessError: Command 'b'find /usr/share /usr/local /etc /opt -name "modules.sh" -o -name "bash" -path "*/init/bash" 2>/dev/null | grep "init/bash\\|modules.sh"\n'' returned non-zero exit status 1.

In [11]:
%%bash
source /etc/profile.d/modules.sh 2>/dev/null || source /usr/share/modules/init/bash
module list

bash: line 1: /usr/share/modules/init/bash: No such file or directory
bash: line 2: module: command not found


CalledProcessError: Command 'b'source /etc/profile.d/modules.sh 2>/dev/null || source /usr/share/modules/init/bash\nmodule list\n'' returned non-zero exit status 127.

In [4]:
from src.models import load_model_from_checkpoint

def _get_n_control_dims(model: torch.nn.Module) -> int:
    n = getattr(model, "n_control_dims", None)
    if n is None:
        n = getattr(getattr(model, "sde_func", None), "n_control_dims", None)
    return int(n or 0)

models = {}
for label, ckpt_name in MODEL_CHECKPOINTS.items():
    ckpt_path = Path(CHECKPOINT_DIR) / ckpt_name
    if not ckpt_path.exists():
        print(f"  [skip] missing checkpoint {ckpt_path}")
        continue
    model, mclass, _ = load_model_from_checkpoint(str(ckpt_path), device=DEVICE)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    models[label] = model
    print(f"  loaded {label:14s} ({mclass}, n_rois={model.n_rois}, n_control_dims={_get_n_control_dims(model)})")

n_control_dims_by_model = {lbl: _get_n_control_dims(m) for lbl, m in models.items()}

/home/hpc/dsaa/dsaa110h/projects/neuroscience_control/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GeForce GTX 1080 Ti which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/home/hpc/dsaa/dsaa110h/projects/neuroscience_control/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/home/hpc/dsaa/dsaa110h/projects/neuroscience_control/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:435: UserWarning: 
NVIDIA GeForce GTX 1080 Ti with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeFo

AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 4. Optimise Control Inputs for Target FC Steering

For every (architecture, target FC) pair we solve:

$$u^\star = \arg\min_{u \in \mathbb{R}^{m}} \, \mathbb{E}_{z_0, W}\!\left[\, \| \mathrm{FC}(x_u) - \mathrm{FC}^{\text{target}} \|_F^2 \,\right] \;+\; \lambda \|u\|_2^2$$

where $x_u$ is the simulated timeseries under constant control $u$. Gradients flow through `torchsde.sdeint` via `torch.autograd`.

The optimisation uses Adam with mini-batches of `BATCH_SIZE` independent noise realisations per step, resampling the initial state and noise each iteration.

In [ ]:
def _simulate(model, u, z0, n_steps):
    """Run a rollout with the batch of controls u (B, n_control_dims)."""
    return model.forward(z0, n_steps=n_steps, dt=DT, dt_min=DT_MIN, control=u)

def _fc_loss(sim_ts, target_fc):
    fc = torch.stack([compute_fc_from_timeseries(s.unsqueeze(0))[0] for s in sim_ts])
    # Frobenius on off-diagonal to avoid the trivially-1 diagonal dominating
    diff = fc - target_fc.unsqueeze(0).to(fc.device)
    mask = 1 - torch.eye(fc.shape[-1], device=fc.device)
    return ((diff * mask) ** 2).sum(dim=(1, 2)).mean()

def optimise_control(model, target_fc, n_control_dims, n_rois,
                     n_opt_steps=N_OPT_STEPS, n_sim_steps=N_SIM_STEPS,
                     batch_size=BATCH_SIZE, lr=LR, reg_l2=REG_L2, seed=0):
    torch.manual_seed(seed)
    u = torch.nn.Parameter(torch.zeros(1, n_control_dims, device=DEVICE))
    opt = torch.optim.Adam([u], lr=lr)
    target = target_fc.to(DEVICE)
    history = []
    for step in range(n_opt_steps):
        z0 = torch.randn(batch_size, n_rois, dtype=torch.complex64, device=DEVICE) * 0.1
        u_batch = u.expand(batch_size, -1).contiguous()
        sim = _simulate(model, u_batch, z0, n_sim_steps)
        fc_err = _fc_loss(sim, target)
        loss = FC_LOSS_WEIGHT * fc_err + reg_l2 * (u ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(float(fc_err.detach().cpu()))
    return u.detach().cpu().squeeze(0).numpy(), history

print("Optimising controls (this may take several minutes on CPU)...")
start = time.time()
optimised = {}  # {model_label: {cond: {'u': np.ndarray, 'loss_history': [...]}}}
for label, model in models.items():
    optimised[label] = {}
    m_ctrl = n_control_dims_by_model[label]
    for cond in TARGET_CONDITIONS:
        t0 = time.time()
        u_star, hist = optimise_control(
            model, empirical_fc_mean[cond], m_ctrl, model.n_rois,
            seed=RANDOM_SEED + hash(cond) % 1000,
        )
        optimised[label][cond] = {"u": u_star, "loss_history": hist}
        print(f"  {label:14s} | target={cond:8s} | u*={np.round(u_star, 3).tolist()} | fc_err_final={hist[-1]:.4f} | {time.time()-t0:.1f}s")
print(f"Total optimisation time: {time.time()-start:.1f}s")

### 4.1 Loss curves

Each panel shows the optimisation trajectory (off-diagonal FC Frobenius error) for both targets under one architecture. A flat curve with low asymptote ⇒ the architecture can steer to that regime with the learned control coupling.

In [ ]:
labels = list(models.keys())
n = len(labels)
cols = min(3, n)
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows), sharex=True)
axes = np.atleast_1d(axes).flatten()
colors = {"Placebo": "#4C72B0", "LSD": "#DD8452"}
for ax, label in zip(axes, labels):
    for cond in TARGET_CONDITIONS:
        hist = optimised[label][cond]["loss_history"]
        ax.plot(hist, color=colors[cond], label=cond, linewidth=1.5)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.set_xlabel("iteration", fontsize=8)
    ax.set_ylabel(r"off-diag FC error $\|\cdot\|_F^2$", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=7)
for ax in axes[len(labels):]:
    ax.set_visible(False)
fig.suptitle("Control optimisation loss curves", fontsize=12, fontweight="bold")
fig.tight_layout()
for ext, d in [("pdf", OUT_DIR_PDF), ("png", OUT_DIR_PNG), ("svg", OUT_DIR_SVG)]:
    fig.savefig(d / f"control_loss_curves.{ext}", bbox_inches="tight", dpi=200)
plt.show()

## 5. Simulation Fidelity: How Close Do Controlled Simulations Get to the Target FC?

We evaluate each $u^\star$ by running a *fresh*, longer rollout (`N_EVAL_STEPS` = 240, matching the empirical series length) with `N_EVAL_BATCHES × BATCH_SIZE` independent noise realisations. We report:

- **FC correlation** — Pearson correlation between the simulated FC and the target mean FC (off-diagonal).
- **FC MSE (off-diagonal)** — mean-squared error on the off-diagonal entries.
- **FCD KS** — two-sample KS statistic between simulated and empirical dynamic-FC distributions.


In [ ]:
def _upper_off_diag(fc):
    idx = torch.triu_indices(fc.shape[-1], fc.shape[-1], offset=1)
    return fc[..., idx[0], idx[1]]

@torch.no_grad()
def simulate_with_control(model, u_vec, n_rois, n_batches=N_EVAL_BATCHES,
                          batch_size=BATCH_SIZE, n_steps=N_EVAL_STEPS, seed=0):
    """Run n_batches independent rollouts of length n_steps; return (B*n_batches, n_rois, n_steps)."""
    torch.manual_seed(seed)
    m = u_vec.shape[0]
    u = torch.as_tensor(u_vec, device=DEVICE, dtype=torch.float32).view(1, m)
    all_ts = []
    for b in range(n_batches):
        z0 = torch.randn(batch_size, n_rois, dtype=torch.complex64, device=DEVICE) * 0.1
        ub = u.expand(batch_size, -1).contiguous()
        ts = model.forward(z0, n_steps=n_steps, dt=DT, dt_min=DT_MIN, control=ub)
        all_ts.append(ts.detach().cpu())
    return torch.cat(all_ts, dim=0)

def fidelity_metrics(sim_ts, target_fc, empirical_fcd):
    """Return dict of fidelity metrics for a batch of simulated trajectories."""
    fcs = compute_fc_from_timeseries(sim_ts)  # (B, R, R)
    off = _upper_off_diag(fcs)                # (B, n_edges)
    tgt_off = _upper_off_diag(target_fc)      # (n_edges,)
    # FC correlation per trajectory
    a = off - off.mean(dim=1, keepdim=True)
    b = tgt_off - tgt_off.mean()
    num = (a * b).sum(dim=1)
    den = (a.norm(dim=1) * b.norm() + 1e-8)
    fc_corr = (num / den)
    fc_mse = ((off - tgt_off) ** 2).mean(dim=1)
    sim_fcd = fcd_windowed(sim_ts)
    ks_stat, ks_p = ks_2samp(sim_fcd.flatten().numpy(), empirical_fcd.flatten().numpy())
    return {
        "fc_corr_mean": float(fc_corr.mean()),
        "fc_corr_std": float(fc_corr.std(unbiased=False)),
        "fc_mse_mean": float(fc_mse.mean()),
        "fc_mse_std": float(fc_mse.std(unbiased=False)),
        "fcd_ks": float(ks_stat),
        "fcd_ks_pvalue": float(ks_p),
        "sim_fcd": sim_fcd,
    }

print("Evaluating optimised controls on fresh rollouts...")
fidelity = {}
for label, model in models.items():
    fidelity[label] = {}
    for cond in TARGET_CONDITIONS:
        u_star = optimised[label][cond]["u"]
        sim_ts = simulate_with_control(model, u_star, model.n_rois, seed=123 + hash(cond) % 1000)
        metrics = fidelity_metrics(sim_ts, empirical_fc_mean[cond], empirical_fcd_by_cond[cond])
        fidelity[label][cond] = metrics
        print(f"  {label:14s} | target={cond:8s} | FC corr={metrics['fc_corr_mean']:.3f}+/-{metrics['fc_corr_std']:.3f} | FC MSE={metrics['fc_mse_mean']:.4f} | FCD KS={metrics['fcd_ks']:.3f}")

In [ ]:
# Bar plot: FC correlation and FC MSE per (model, target)
metric_cfg = [
    ("fc_corr_mean", "fc_corr_std", "FC correlation ↑", False),
    ("fc_mse_mean", "fc_mse_std", "FC MSE (off-diagonal) ↓", False),
    ("fcd_ks", None, "FCD KS vs. empirical ↓", False),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
x = np.arange(len(labels))
bar_w = 0.38
for ax, (m_mean, m_std, title, _) in zip(axes, metric_cfg):
    for i, cond in enumerate(TARGET_CONDITIONS):
        means = [fidelity[lbl][cond][m_mean] for lbl in labels]
        stds = [fidelity[lbl][cond][m_std] if m_std else 0 for lbl in labels]
        ax.bar(x + (i - 0.5) * bar_w, means, bar_w, yerr=stds, capsize=3,
               color=colors[cond], label=cond, alpha=0.9, edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(fontsize=8)
fig.suptitle("Controlled-simulation fidelity vs. target FC regimes", fontsize=12, fontweight="bold")
fig.tight_layout()
for ext, d in [("pdf", OUT_DIR_PDF), ("png", OUT_DIR_PNG), ("svg", OUT_DIR_SVG)]:
    fig.savefig(d / f"control_fidelity_bars.{ext}", bbox_inches="tight", dpi=200)
plt.show()

## 6. Optimised Control Inputs Across Models

A key question: **do different architectures agree on what $u$ drives the system between Placebo and LSD?** Because each architecture interprets $u$ through its own learned control-coupling matrix, the absolute values of $u^\star$ are not directly comparable. But we can still compare (i) the sign and direction of the *Placebo → LSD* shift $\Delta u = u^\star_{\text{LSD}} - u^\star_{\text{Placebo}}$ and (ii) the magnitudes.

In [ ]:
# Build a table of u* per (model, target). Pad 1-D models to the max m for display.
m_max = max(n_control_dims_by_model.values())
rows_list = []
for label in labels:
    u_p = optimised[label]["Placebo"]["u"]
    u_l = optimised[label]["LSD"]["u"]
    du = u_l - u_p
    def _pad(v):
        out = np.full(m_max, np.nan)
        out[: v.shape[0]] = v
        return out
    rows_list.append({
        "Model": label,
        "n_ctrl": n_control_dims_by_model[label],
        **{f"u_Placebo[{i}]": float(v) for i, v in enumerate(_pad(u_p))},
        **{f"u_LSD[{i}]": float(v) for i, v in enumerate(_pad(u_l))},
        **{f"delta_u[{i}]": float(v) for i, v in enumerate(_pad(du))},
        "||delta_u||": float(np.linalg.norm(du)),
    })
import pandas as pd
df_u = pd.DataFrame(rows_list).set_index("Model")
df_u.round(3)

In [ ]:
# Visualise u* as grouped bars. For models with m=1, only the first dimension exists.
fig, axes = plt.subplots(1, m_max, figsize=(5 * m_max, 4), squeeze=False)
for dim in range(m_max):
    ax = axes[0, dim]
    vals_p, vals_l = [], []
    for label in labels:
        u_p = optimised[label]["Placebo"]["u"]
        u_l = optimised[label]["LSD"]["u"]
        vals_p.append(u_p[dim] if dim < u_p.shape[0] else np.nan)
        vals_l.append(u_l[dim] if dim < u_l.shape[0] else np.nan)
    x = np.arange(len(labels))
    ax.bar(x - 0.2, vals_p, 0.4, color=colors["Placebo"], label="Placebo", edgecolor="white")
    ax.bar(x + 0.2, vals_l, 0.4, color=colors["LSD"], label="LSD", edgecolor="white")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.set_title(f"$u^\\star$ component {dim}", fontsize=10, fontweight="bold")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.legend(fontsize=8)
fig.suptitle("Optimised control inputs $u^\\star$ per architecture and target", fontsize=12, fontweight="bold")
fig.tight_layout()
for ext, d in [("pdf", OUT_DIR_PDF), ("png", OUT_DIR_PNG), ("svg", OUT_DIR_SVG)]:
    fig.savefig(d / f"control_u_star.{ext}", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# Pairwise cosine similarity between delta-u across models with matching m.
labels_m1 = [l for l in labels if n_control_dims_by_model[l] == 1]
labels_by_m = {m: [l for l in labels if n_control_dims_by_model[l] == m] for m in set(n_control_dims_by_model.values())}

def _pairwise_cosine(vecs):
    V = np.stack(vecs, axis=0)
    N = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
    return N @ N.T

for m_dim, lbls in labels_by_m.items():
    if len(lbls) < 2:
        continue
    dus = [optimised[l]["LSD"]["u"] - optimised[l]["Placebo"]["u"] for l in lbls]
    if m_dim == 1:
        # Cosine on a scalar is just sign agreement
        sim = np.sign(np.array(dus).squeeze()[:, None] * np.array(dus).squeeze()[None, :])
        title = r"sign agreement of $\Delta u$ (m=1 models)"
    else:
        sim = _pairwise_cosine(dus)
        title = r"cosine similarity of $\Delta u$ (m=%d models)" % m_dim
    fig, ax = plt.subplots(figsize=(1.5 + 0.6 * len(lbls), 1.5 + 0.6 * len(lbls)))
    im = ax.imshow(sim, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(lbls))); ax.set_yticks(range(len(lbls)))
    ax.set_xticklabels(lbls, rotation=30, ha="right", fontsize=8)
    ax.set_yticklabels(lbls, fontsize=8)
    for i in range(len(lbls)):
        for j in range(len(lbls)):
            ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=7, color="black")
    ax.set_title(title, fontsize=10, fontweight="bold")
    fig.colorbar(im, ax=ax, shrink=0.7)
    fig.tight_layout()
    for ext, d in [("pdf", OUT_DIR_PDF), ("png", OUT_DIR_PNG), ("svg", OUT_DIR_SVG)]:
        fig.savefig(d / f"control_delta_u_similarity_m{m_dim}.{ext}", bbox_inches="tight", dpi=200)
    plt.show()

## 7. Statistical Indistinguishability on FC/FCD

Before probing *transfer*, we identify which architectures are statistically indistinguishable on simulated FC and FCD under their **matched** control (i.e., $u^\star_{\text{LSD}}$ for the LSD target). We run the two-sample Kolmogorov–Smirnov test on:

- **FC edges** — pooled off-diagonal entries across the `N_EVAL_BATCHES * BATCH_SIZE` rollouts;
- **FCD** — pooled dynamic-FC upper-triangle entries.

A **high p-value** (≥ 0.05) means we cannot reject the null that the two distributions are the same; those pairs are candidates for control-input transfer, because they produce dynamics that a statistical observer cannot separate.

In [ ]:
# Collect FC-edge and FCD samples per model for the LSD target
target_cond = "LSD"
fc_samples = {}
fcd_samples = {}
for label, model in models.items():
    sim_ts = simulate_with_control(model, optimised[label][target_cond]["u"], model.n_rois, seed=777)
    fcs = compute_fc_from_timeseries(sim_ts)
    fc_samples[label] = _upper_off_diag(fcs).flatten().numpy()
    fcd_samples[label] = fcd_windowed(sim_ts).flatten().numpy()
    print(f"  {label}: {fc_samples[label].size} FC edges, {fcd_samples[label].size} FCD samples")

def pairwise_ks(samples_by_model):
    lbls = list(samples_by_model.keys())
    p = np.ones((len(lbls), len(lbls)))
    ks = np.zeros((len(lbls), len(lbls)))
    for i, a in enumerate(lbls):
        for j, b in enumerate(lbls):
            if i == j:
                continue
            stat, pval = ks_2samp(samples_by_model[a], samples_by_model[b])
            ks[i, j] = stat; p[i, j] = pval
    return lbls, ks, p

lbls_fc, fc_ks, fc_p = pairwise_ks(fc_samples)
lbls_fcd, fcd_ks, fcd_p = pairwise_ks(fcd_samples)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (ks_mat, p_mat, title, lbls) in zip(
    axes,
    [(fc_ks, fc_p, "FC edges — KS statistic (p-value annotated)", lbls_fc),
     (fcd_ks, fcd_p, "FCD — KS statistic (p-value annotated)", lbls_fcd)],
):
    im = ax.imshow(ks_mat, cmap="viridis")
    ax.set_xticks(range(len(lbls))); ax.set_yticks(range(len(lbls)))
    ax.set_xticklabels(lbls, rotation=30, ha="right", fontsize=8)
    ax.set_yticklabels(lbls, fontsize=8)
    ax.set_title(title, fontsize=10, fontweight="bold")
    for i in range(len(lbls)):
        for j in range(len(lbls)):
            if i == j:
                ax.text(j, i, "—", ha="center", va="center", fontsize=7, color="white")
            else:
                color = "white" if ks_mat[i, j] > ks_mat.max() * 0.5 else "black"
                ax.text(j, i, f"ks={ks_mat[i,j]:.2f}\np={p_mat[i,j]:.0e}", ha="center", va="center", fontsize=6, color=color)
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle("Pairwise statistical (in)distinguishability between architectures under LSD control", fontsize=12, fontweight="bold")
fig.tight_layout()
for ext, d in [("pdf", OUT_DIR_PDF), ("png", OUT_DIR_PNG), ("svg", OUT_DIR_SVG)]:
    fig.savefig(d / f"control_pairwise_ks.{ext}", bbox_inches="tight", dpi=200)
plt.show()

# Identify indistinguishable pairs at alpha=0.05 on BOTH FC and FCD
ALPHA = 0.05
indist_pairs = []
for i, a in enumerate(lbls_fc):
    for j, b in enumerate(lbls_fc):
        if j <= i:
            continue
        if fc_p[i, j] >= ALPHA and fcd_p[i, j] >= ALPHA:
            indist_pairs.append((a, b, float(fc_p[i, j]), float(fcd_p[i, j])))
print("\nStatistically indistinguishable pairs (p≥%.2f on BOTH FC and FCD):" % ALPHA)
if not indist_pairs:
    print("  (none)")
for a, b, pfc, pfcd in indist_pairs:
    print(f"  {a} ↔ {b}: p_FC={pfc:.3f}, p_FCD={pfcd:.3f}")

## 8. Transferring Controls Between Architectures

We now apply each $u^\star$ optimised on a **source** architecture to a **target** architecture and re-simulate. If the target model produces FC close to the desired regime, the control is *transferable*.

Only pairs with matching `n_control_dims` are compatible without projection; we report those separately. For each (source → target, regime) we record the FC correlation with the empirical mean FC of the regime, and the *drop* vs. the matched (source==target) optimum.

In [ ]:
transfer = {cond: {} for cond in TARGET_CONDITIONS}
for cond in TARGET_CONDITIONS:
    for m_dim, lbls in labels_by_m.items():
        if len(lbls) < 2:
            continue
        mat = np.full((len(lbls), len(lbls)), np.nan)
        for i, src in enumerate(lbls):
            u_src = optimised[src][cond]["u"]
            for j, tgt in enumerate(lbls):
                sim_ts = simulate_with_control(
                    models[tgt], u_src, models[tgt].n_rois, seed=999 + i * 100 + j
                )
                m = fidelity_metrics(sim_ts, empirical_fc_mean[cond], empirical_fcd_by_cond[cond])
                mat[i, j] = m["fc_corr_mean"]
        transfer[cond][m_dim] = {"labels": lbls, "fc_corr": mat}
        print(f"Transfer matrix ({cond}, m={m_dim}):")
        print(pd.DataFrame(mat, index=lbls, columns=lbls).round(3))
        print()

In [ ]:
# Plot transfer matrices (FC correlation) side by side for Placebo and LSD, matching-m groups.
m_dims_plotted = [m for m, lbls in labels_by_m.items() if len(lbls) >= 2]
ncols = len(TARGET_CONDITIONS)
nrows = len(m_dims_plotted)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows), squeeze=False)
for r, m_dim in enumerate(m_dims_plotted):
    lbls = labels_by_m[m_dim]
    for c, cond in enumerate(TARGET_CONDITIONS):
        ax = axes[r, c]
        mat = transfer[cond][m_dim]["fc_corr"]
        im = ax.imshow(mat, cmap="RdYlGn", vmin=-0.5, vmax=1.0)
        ax.set_xticks(range(len(lbls))); ax.set_yticks(range(len(lbls)))
        ax.set_xticklabels(lbls, rotation=30, ha="right", fontsize=8)
        ax.set_yticklabels(lbls, fontsize=8)
        ax.set_xlabel("target architecture", fontsize=9)
        ax.set_ylabel("source architecture (u from)", fontsize=9)
        ax.set_title(f"{cond} target | m={m_dim}", fontsize=10, fontweight="bold")
        for i in range(len(lbls)):
            for j in range(len(lbls)):
                v = mat[i, j]
                if not np.isnan(v):
                    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                            color="black" if v > 0.4 else "white", fontweight="bold" if i == j else "normal")
        fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle("Cross-architecture transfer of optimised controls (FC correlation to target)", fontsize=12, fontweight="bold")
fig.tight_layout()
for ext, d in [("pdf", OUT_DIR_PDF), ("png", OUT_DIR_PNG), ("svg", OUT_DIR_SVG)]:
    fig.savefig(d / f"control_transfer_matrix.{ext}", bbox_inches="tight", dpi=200)
plt.show()

In [ ]:
# Focus: how well does transfer work *specifically* on statistically indistinguishable pairs?
if indist_pairs:
    rows = []
    for cond in TARGET_CONDITIONS:
        for a, b, pfc, pfcd in indist_pairs:
            m_dim = n_control_dims_by_model[a]
            if n_control_dims_by_model[b] != m_dim:
                continue
            lbls = transfer[cond][m_dim]["labels"]
            i_a, i_b = lbls.index(a), lbls.index(b)
            mat = transfer[cond][m_dim]["fc_corr"]
            rows.append({
                "target_cond": cond,
                "pair": f"{a} ↔ {b}",
                "matched_AA": mat[i_a, i_a],
                "matched_BB": mat[i_b, i_b],
                "transfer_A→B": mat[i_a, i_b],
                "transfer_B→A": mat[i_b, i_a],
                "drop_A→B": mat[i_a, i_a] - mat[i_a, i_b],
                "drop_B→A": mat[i_b, i_b] - mat[i_b, i_a],
                "p_FC": pfc,
                "p_FCD": pfcd,
            })
    df_trans = pd.DataFrame(rows)
    print("\nTransfer performance on statistically indistinguishable pairs:")
    display(df_trans.round(3))
else:
    print("No statistically indistinguishable pairs to highlight.")

## 9. Save Results

The numerical results are dumped to `results/lsd_in_silico_control.json` so they can be cited/plotted from the paper without re-running the optimisation.

In [ ]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "dataset_type": DATASET_TYPE,
    "target_conditions": TARGET_CONDITIONS,
    "n_control_dims": n_control_dims_by_model,
    "optimised": {
        label: {cond: {"u": v["u"].tolist(), "loss_history": v["loss_history"]}
                for cond, v in per_cond.items()}
        for label, per_cond in optimised.items()
    },
    "fidelity": {
        label: {cond: {k: v for k, v in m.items() if k != "sim_fcd"}
                for cond, m in per_cond.items()}
        for label, per_cond in fidelity.items()
    },
    "transfer": {
        cond: {
            str(m_dim): {
                "labels": dat["labels"],
                "fc_corr_matrix": dat["fc_corr"].tolist(),
            }
            for m_dim, dat in by_m.items()
        }
        for cond, by_m in transfer.items()
    },
    "indistinguishable_pairs": [
        {"a": a, "b": b, "p_FC": p_fc, "p_FCD": p_fcd}
        for a, b, p_fc, p_fcd in indist_pairs
    ],
    "config": {
        "dt": DT, "dt_min": DT_MIN, "n_sim_steps": N_SIM_STEPS,
        "batch_size": BATCH_SIZE, "n_opt_steps": N_OPT_STEPS, "lr": LR,
        "reg_l2": REG_L2, "n_eval_batches": N_EVAL_BATCHES, "n_eval_steps": N_EVAL_STEPS,
    },
}
with open(RESULTS_PATH, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved results to {RESULTS_PATH}")

## Takeaways for the Paper

- **Steerability.** Every architecture can be driven from the Placebo FC regime to the LSD FC regime by a single optimised constant control vector, but the *fidelity* of the steered simulation (Section 5) differs across architectures.
- **Agreement on direction.** The sign / cosine similarity of the Placebo → LSD shift $\Delta u$ (Section 6) tells us whether the architectures agree on *which way to push* the control, even when they disagree on magnitude.
- **Transfer robustness.** The cross-architecture transfer matrix (Section 8) reveals the cost of model mismatch: diagonal entries (matched control) bound the off-diagonal entries from above. Architectures that are statistically indistinguishable on FC/FCD (Section 7) tend to exhibit the smallest transfer drop, supporting the claim that FC/FCD-equivalent models are also *control-equivalent*.